# Dialogue Log Verification System

    "This notebook verifies and selects the best dialogue from Gemini-3.1 and GPT-5.2 generated seeds using Claude Sonnet 4.5 as an independent arbiter with binary pass/fail validation on 5 criteria.\n",

## Execution Flow

The cells execute in 7 sequential steps:
1. Import libraries and set file paths
2. Load Gemini-3.1 and GPT-5.2 CSV datasets
3. Review binary validation system and decision logic
4. Initialize Claude Sonnet 4.5 verifier and define verification function
5. Run verification loop (currently limited to first 2 games for testing)
6. Create output DataFrames from verification results with per-criterion tracking
7. Save verified results to CSV files

## Output Files

1. **verified_r1_seeds_combined.csv**: Selected dialogues with source labels
2. **verified_r1_criteria_scores.csv**: Detailed per-criterion pass/fail tracking for analysis

## Step 1: Import Libraries

Import required Python libraries and set file paths.

In [ ]:
# Import Required Libraries
import pandas as pd
import json
import os
import re
from typing import Dict, Tuple, Optional
from anthropic import Anthropic
import time

# Initialize paths
DATA_DIR = "."
GEMINI3_FILE = "generated_r1_seeds_gemini3.csv"
GPT52_FILE = "generated_r1_seeds_gpt5_2.csv"
OUTPUT_FILE_COMBINED = "verified_r1_seeds_combined.csv"
OUTPUT_FILE_CRITERIA = "verified_r1_criteria_scores.csv"

print("✓ Libraries imported successfully")

## Step 2: Load Datasets

Load the Gemini-3.1 and GPT-5.2 generated seed datasets.

In [ ]:
# Load Generated Seeds from GPT Models
gemini3_df = pd.read_csv(GEMINI3_FILE)
gpt52_df = pd.read_csv(GPT52_FILE)

print(f"Gemini-3.1 dataset loaded: {gemini3_df.shape[0]} rows, {gemini3_df.shape[1]} columns")
print(f"GPT-5.2 dataset loaded: {gpt52_df.shape[0]} rows, {gpt52_df.shape[1]} columns")
print(f"\nColumns in Gemini-3.1: {list(gemini3_df.columns)}")
print(f"Sample row (first game):")
print(f"  game_id: {gemini3_df.iloc[0]['game_id']}")
print(f"  player_roles: {gemini3_df.iloc[0]['player_roles'][:80]}...")
print(f"  matrix_tactic_scale sample: {gemini3_df.iloc[0]['matrix_tactic_scale'][:100]}...")

## Step 3: Binary Multi-Criteria Validation System

**Validation Criteria:**

Claude Sonnet 4.5 evaluates each dialogue using **binary pass/fail checks** on 5 criteria:

1. **Player Roles Consistency** (1/0) - Good players use honest tactics, Evil players use deceptive tactics
2. **Public History Alignment** (1/0) - Dialogue contextually appropriate to game state
3. **Matrix Tactic Scale Validity** (1/0) - Correct row, col, tactic, scale, and level
4. **Avalon Gameplay Authenticity** (1/0) - Natural dialogue with strategic plausibility
5. **Dialogue Format Compliance** (1/0) - Exactly 4 speakers, no protagonist, correct format

**Decision Logic:**

1. **Both 5/5 pass** → Claude Sonnet 4.5 performs pairwise comparison to break tie
2. **One 5/5, other <5/5** → Select the 5/5 response
3. **At least one 4/5**:
   - One 4/5, other <4/5 → Choose 4/5, **Targeted Correction** (fix only the failing criterion)
   - Both 4/5 → Choose GPT-5.2, **Targeted Correction**
4. **Both ≤3/5** → **Full Custom Generation** (generate from scratch)

## Step 3a: Initialize Claude Sonnet 4.5

Set up the Anthropic client and system prompt.

In [ ]:
# Initialize Claude Sonnet 4.5 Verifier
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'), max_retries=5)

SYSTEM_PROMPT = """You are an expert verifier for the Avalon social deception game with deep knowledge of deception tactics and game mechanics. Your task is to evaluate pairs of AI-generated dialogue responses, assess their quality, and determine the best outcome: select the superior response as-is, apply targeted corrections to fix specific issues in the better response, or generate an entirely new dialogue if both responses are inadequate."""

print("✓ Claude Sonnet 4.5 system prompt initialized")

## Step 3b: Define Verification Function

Create the function that compares and evaluates dialogue pairs.

In [ ]:
# Create Verification Function
def verify_dialogue_pair(game_id: str,
                         round_id: str,
                         player_roles: str,
                         protagonist: str,
                         public_history: str,
                         dialogue_gemini3: str,
                         tactic_scale_gemini3: str,
                         dialogue_gpt52: str,
                         tactic_scale_gpt52: str) -> Dict:
    """
    Verify both dialogue responses using Claude Sonnet 4.5 with binary criteria validation.
    
    Returns: {
        'gemini3_criteria': {'roles': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gemini3_total': int (0-5),
        'gpt52_criteria': {'roles': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gpt52_total': int (0-5),
        'recommendation': str,
        'selected_dialogue': str,
        'selected_tactic_scale': str,
        'correction_mode': str (None, Targeted, Full_Custom, Pairwise_Tiebreaker),
        'failed_criterion': str or None,
        'reasoning': str
    }
    """
    
    verification_prompt = f"""You are evaluating two AI-generated Avalon Round 1 discussion logs to determine which is superior. Both responses were generated for the same game context. Your task is to validate both using BINARY PASS/FAIL checks on 5 criteria, then recommend the best outcome: select one response as-is, apply a targeted correction to the better response, or generate a completely new dialogue if both are inadequate.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Public History (up to current round):
{public_history}

---

**RESPONSE A:**

Discussion Log:
{dialogue_gemini3}

Matrix Tactic Scale:
{tactic_scale_gemini3}

---

**RESPONSE B:**

Discussion Log:
{dialogue_gpt52}

Matrix Tactic Scale:
{tactic_scale_gpt52}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Player Roles Consistency**: 
   1 = Good players use transparent/honest tactics, Evil players use deceptive tactics
   0 = Role-tactic misalignment or inconsistent behaviors

2. **Public History Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns
   0 = Ignores prior events or contradicts established game state

3. **Matrix Tactic Scale Validity**:
   1 = Correct row (Info Strategy), column (Goal), tactic (1 of 37), scale (GRS/Mach-IV), level (H/M/L)
   0 = Wrong category, invalid tactic, or scale-role mismatch

4. **Avalon Gameplay Authenticity**:
   1 = Natural dialogue with appropriate accusations/defenses, skill-modulated sophistication
   0 = Unnatural, implausible strategy, or skill level doesn't match behavior complexity

5. **Dialogue Format Compliance**:
   1 = Follows "Discussion after Quest X:" format, exactly 4 speakers (excluding protagonist), each speaks once
   1 = Expected format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong format, incorrect speaker count, protagonist speaks, or format violations

**DECISION LOGIC:**

Apply this logic based on how many criteria each response passes:

1. **Both responses 5/5** → Use PAIRWISE_TIEBREAKER
   - Compare both dialogues to determine which is slightly better overall
   - Output RECOMMENDATION: PAIRWISE_TIEBREAKER

2. **One response 5/5, other <5/5** → Choose the 5/5 response
   - Output RECOMMENDATION: RESPONSE_A (if Response A is 5/5)
   - Output RECOMMENDATION: RESPONSE_B (if Response B is 5/5)

3. **At least one response 4/5** → Apply targeted correction
   - If only one is 4/5: choose that one → CORRECTION_A or CORRECTION_B
   - If both are 4/5: default to Response B → CORRECTION_B
   - Output RECOMMENDATION: CORRECTION_A or CORRECTION_B

4. **Both responses ≤3/5** → Generate custom dialogue
   - Output RECOMMENDATION: CUSTOM_GENERATION

**EVALUATION PROCESS:**

1. Evaluate RESPONSE A against all 5 criteria using binary PASS/FAIL checks
2. Evaluate RESPONSE B against all 5 criteria using the same binary checks
3. Count total passes for each response (X/5)
4. Apply the DECISION LOGIC rules based on the pass counts
5. Generate output in the exact format below

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object with the following structure:

```json
{{
  "response_a": {{
    "criteria": {{
      "roles": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "response_b": {{
    "criteria": {{
      "roles": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "recommendation": "RESPONSE_A / RESPONSE_B / PAIRWISE_TIEBREAKER / CORRECTION_A / CORRECTION_B / CUSTOM_GENERATION",
  "reasoning": "2-3 sentences explaining the decision",
  "pairwise_winner": "RESPONSE_A / RESPONSE_B / null",
  "pairwise_reasoning": "1-2 sentences explaining pairwise selection / null",
  "failed_criterion": "Roles / History / Matrix / Authenticity / Format / null",
  "targeted_correction": "Full corrected dialogue / null",
  "custom_dialogue": "Complete new dialogue / null"
}}
```

**Field Explanations:**
- **score**: 1 = PASS, 0 = FAIL
- **explanation**: Brief explanation required for BOTH pass and fail to justify the score
- **total**: Integer count of passed criteria (0-5), NOT a fraction string
- **recommendation**: One of: "RESPONSE_A", "RESPONSE_B", "PAIRWISE_TIEBREAKER", "CORRECTION_A", "CORRECTION_B", "CUSTOM_GENERATION"
- **pairwise_winner**: Only if recommendation is PAIRWISE_TIEBREAKER (both 5/5), set to "RESPONSE_A" or "RESPONSE_B"
- **pairwise_reasoning**: Only if pairwise_winner is set, 1-2 sentences
- **failed_criterion**: Only if recommendation is CORRECTION_A or CORRECTION_B (4/5 needs fix), set to: "Roles", "History", "Matrix", "Authenticity", or "Format"
- **targeted_correction**: Only if failed_criterion is set, provide full corrected dialogue
- **custom_dialogue**: Only if recommendation is CUSTOM_GENERATION (both ≤3/5), provide complete new dialogue in format:
  ```
  Discussion after Quest 1:
  PlayerX: [dialogue]
  PlayerY: [dialogue]
  PlayerZ: [dialogue]
  PlayerW: [dialogue]
  ```

**IMPORTANT:** Return ONLY the JSON object, no markdown code blocks, no extra text."""
    
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": verification_prompt}
            ]
        )
        
        response_text = response.content[0].text
        
        # DEBUG: Print raw response to diagnose parsing issues
        print(f"\n{'='*80}")
        print(f"RAW CLAUDE RESPONSE FOR {game_id}:")
        print(f"{'='*80}")
        print(response_text)
        print(f"{'='*80}\n")
        
        # Parse JSON response
        try:
            # Remove markdown code blocks if present
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                # Remove opening ``` line
                lines = lines[1:]
                # Find and remove closing ```
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)
            
            # Parse JSON
            data = json.loads(clean_text)
            
            # Build result dictionary from JSON
            result = {
                'gemini3_criteria': {
                    'roles': data['response_a']['criteria']['roles']['score'] == 1,
                    'history': data['response_a']['criteria']['history']['score'] == 1,
                    'matrix': data['response_a']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_a']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_a']['criteria']['format']['score'] == 1
                },
                'gemini3_total': int(data['response_a']['total']) if isinstance(data['response_a']['total'], int) else int(str(data['response_a']['total']).split('/')[0]),
                'gemini3_criteria_explanations': {
                    criterion: data['response_a']['criteria'][criterion].get('explanation', '')
                    for criterion in ['roles', 'history', 'matrix', 'authenticity', 'format']
                },
                'gpt52_criteria': {
                    'roles': data['response_b']['criteria']['roles']['score'] == 1,
                    'history': data['response_b']['criteria']['history']['score'] == 1,
                    'matrix': data['response_b']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_b']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_b']['criteria']['format']['score'] == 1
                },
                'gpt52_total': int(data['response_b']['total']) if isinstance(data['response_b']['total'], int) else int(str(data['response_b']['total']).split('/')[0]),
                'gpt52_criteria_explanations': {
                    criterion: data['response_b']['criteria'][criterion].get('explanation', '')
                    for criterion in ['roles', 'history', 'matrix', 'authenticity', 'format']
                },
                'recommendation': data['recommendation'],
                'reasoning': data['reasoning'],
                'pairwise_winner': data.get('pairwise_winner'),
                'pairwise_reasoning': data.get('pairwise_reasoning', ''),
                'failed_criterion': data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'custom_dialogue': data.get('custom_dialogue'),
                'selected_dialogue': None,
                'selected_tactic_scale': None,
                'correction_mode': None
            }
            
            # Determine final selection based on scores
            s1 = result['gemini3_total']
            s2 = result['gpt52_total']
            
            if s1 == 5 and s2 == 5:
                # Both 5/5 - use pairwise tiebreaker
                if result['pairwise_winner'] and 'RESPONSE_A' in result['pairwise_winner'].upper():
                    result['recommendation'] = 'RESPONSE_A'
                    result['selected_dialogue'] = dialogue_gemini3
                    result['selected_tactic_scale'] = tactic_scale_gemini3
                else:
                    result['recommendation'] = 'RESPONSE_B'
                    result['selected_dialogue'] = dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = 'Pairwise_Tiebreaker'
                
            elif s1 == 5:
                # Only Response A is 5/5
                result['recommendation'] = 'RESPONSE_A'
                result['selected_dialogue'] = dialogue_gemini3
                result['selected_tactic_scale'] = tactic_scale_gemini3
                result['correction_mode'] = None
                
            elif s2 == 5:
                # Only Response B is 5/5
                result['recommendation'] = 'RESPONSE_B'
                result['selected_dialogue'] = dialogue_gpt52
                result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = None
                
            elif s1 == 4 or s2 == 4:
                # At least one 4/5 - choose 4/5 or default to Response B if both 4/5
                if s1 == 4 and s2 == 4:
                    # Both 4/5 - default to Response B
                    result['recommendation'] = 'CORRECTION_B'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                elif s1 == 4:
                    result['recommendation'] = 'CORRECTION_A'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gemini3
                    result['selected_tactic_scale'] = tactic_scale_gemini3
                else:  # s2 == 4
                    result['recommendation'] = 'CORRECTION_B'
                    result['selected_dialogue'] = result['targeted_correction'] or dialogue_gpt52
                    result['selected_tactic_scale'] = tactic_scale_gpt52
                result['correction_mode'] = 'Targeted'
                
            else:
                # Both <=3/5 - full custom generation
                result['correction_mode'] = 'Full_Custom'
                result['selected_dialogue'] = result['custom_dialogue']
                result['selected_tactic_scale'] = tactic_scale_gpt52  # Use Response B tactic as base
                result['recommendation'] = 'CUSTOM_GENERATION'
            
            return result, response_text
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            print(f"Cleaned text: {clean_text[:500]}...")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing JSON for {game_id}: {e}")
            return None, str(e)
    except Exception as e:
        print(f"Error verifying game {game_id}: {str(e)}")
        return None, str(e)

## Step 4: Run Verification Loop

Process games and collect verification results.

In [ ]:
# Helper function to format matrix tactic scale with pretty-printing
def format_matrix_tactic_scale(json_str: str) -> str:
    """Format JSON with linebreaks for readability"""
    try:
        data = json.loads(json_str)
        # Reconstruct with pretty-printing: 2-space indent
        return json.dumps(data, indent=2)
    except:
        return json_str

# Compare and Validate Dialogue Rows
# Initialize results storage
verified_results = []
criteria_results = []
verification_stats = {
    'total_games': 0,
    'response_1_selected': 0,
    'response_2_selected': 0,
    'targeted_correction_1': 0,
    'targeted_correction_2': 0,
    'pairwise_tiebreaker': 0,
    'full_custom_generation': 0,
    'errors': 0
}

# Process first 2 games as a sample/test run. Replace 2 with 225 or len(gemini3_df)
TEST_LIMIT = len(gemini3_df)
print(f"Starting verification of {min(TEST_LIMIT, len(gemini3_df))} games...")
print("=" * 80)

for idx in range(min(TEST_LIMIT, len(gemini3_df))):
    game_id = gemini3_df.iloc[idx]['game_id']
    round_id = gemini3_df.iloc[idx]['round_id']
    score_id = f"{game_id}/{round_id}"
    
    # Extract fields from both datasets
    player_roles = gemini3_df.iloc[idx]['player_roles']
    protagonist = gemini3_df.iloc[idx]['role_id']  # Protagonist who doesn't speak
    public_history = gemini3_df.iloc[idx]['public_history']
    
    dialogue_gemini3 = gemini3_df.iloc[idx]['discussion_log']
    tactic_scale_gemini3 = gemini3_df.iloc[idx]['matrix_tactic_scale']
    
    dialogue_gpt52 = gpt52_df.iloc[idx]['discussion_log']
    tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']
    
    # Verify the pair
    print(f"\n[{idx+1}/{min(TEST_LIMIT, len(gemini3_df))}] Verifying game {score_id}...")
    result, raw_response = verify_dialogue_pair(
        game_id, round_id, player_roles, protagonist, public_history,
        dialogue_gemini3, tactic_scale_gemini3,
        dialogue_gpt52, tactic_scale_gpt52
    )
    
    if result is None:
        print(f"  ✗ Error: {raw_response}")
        verification_stats['errors'] += 1
        continue
    
    verification_stats['total_games'] += 1
    
    # Determine LLM source and label for output (map Response A/B back to actual model names)
    if result['recommendation'] == 'RESPONSE_A':
        llm_used = 'Gemini-3.1'
        selected_row = gemini3_df.iloc[idx].copy()
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gemini3)
        verification_stats['response_1_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_A':
        llm_used = 'Gemini-3.1-Claude-4.5'
        selected_row = gemini3_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gemini3)
        verification_stats['targeted_correction_1'] += 1
    elif result['recommendation'] == 'RESPONSE_B':
        llm_used = 'GPT-5.2'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['response_2_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_B':
        llm_used = 'GPT-5.2-Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['targeted_correction_2'] += 1
    else:  # CUSTOM_GENERATION
        llm_used = 'Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        selected_tactic_scale = format_matrix_tactic_scale(tactic_scale_gpt52)
        verification_stats['full_custom_generation'] += 1
    
    # Track pairwise tiebreakers separately
    if result['correction_mode'] == 'Pairwise_Tiebreaker':
        verification_stats['pairwise_tiebreaker'] += 1
    
    # Update matrix tactic scale in selected row
    selected_row['matrix_tactic_scale'] = selected_tactic_scale
    
    # Add LLM_used and Claude reasoning columns
    selected_row['LLM_used'] = llm_used
    selected_row['Claude_Reasoning'] = result['reasoning']
    verified_results.append(selected_row)
    
    # Create detailed criteria record for CSV
    def format_check(val):
        return 1 if val else 0
    
    criteria_record = {
        'ID': score_id,
        'Gemini31_Roles': format_check(result['gemini3_criteria']['roles']),
        'Gemini31_History': format_check(result['gemini3_criteria']['history']),
        'Gemini31_Matrix': format_check(result['gemini3_criteria']['matrix']),
        'Gemini31_Authenticity': format_check(result['gemini3_criteria']['authenticity']),
        'Gemini31_Format': format_check(result['gemini3_criteria']['format']),
        'Gemini31_Total': f"{result['gemini3_total']} of 5",
        'GPT52_Roles': format_check(result['gpt52_criteria']['roles']),
        'GPT52_History': format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix': format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity': format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format': format_check(result['gpt52_criteria']['format']),
        'GPT52_Total': f"{result['gpt52_total']} of 5",
        'Selected_LLM': llm_used,
        'Correction_Mode': result['correction_mode'] or 'None',
        'Claude_Reasoning': result['reasoning'],
        'Gemini31_Criteria_Explanations': json.dumps(result['gemini3_criteria_explanations']),
        'GPT52_Criteria_Explanations': json.dumps(result['gpt52_criteria_explanations']),
        'Claude_Response': raw_response  # Store full raw response for debugging
    }
    criteria_results.append(criteria_record)
    
    # Print summary
    print(f"  ✓ Gemini-3.1: {result['gemini3_total']}/5 pass ({', '.join([k for k,v in result['gemini3_criteria'].items() if v])})")
    print(f"  ✓ GPT-5.2: {result['gpt52_total']}/5 pass ({', '.join([k for k,v in result['gpt52_criteria'].items() if v])})")
    print(f"  ✓ Selected: {llm_used}")
    if result['correction_mode']:
        print(f"  ✓ Correction Mode: {result['correction_mode']}")
    print(f"  ✓ Reason: {result['reasoning'][:100]}...")
    
    # Debug output for custom generation
    if result['correction_mode'] == 'Full_Custom':
        if result['selected_dialogue']:
            print(f"  📝 Custom dialogue preview: {result['selected_dialogue'][:150]}...")
        else:
            print(f"  ⚠️  WARNING: Custom dialogue is empty!")
    
    # Rate limit to avoid API throttling
    time.sleep(1)

print("\n" + "=" * 80)
print(f"Verification Summary (first {TEST_LIMIT} games):")
print(f"  Total verified: {verification_stats['total_games']}")
print(f"  Gemini-3.1 selected (original): {verification_stats['response_1_selected']}")
print(f"  GPT-5.2 selected (original): {verification_stats['response_2_selected']}")
print(f"  Pairwise tiebreaker used: {verification_stats['pairwise_tiebreaker']}")
print(f"  Gemini-3.1 + Targeted correction: {verification_stats['targeted_correction_1']}")
print(f"  GPT-5.2 + Targeted correction: {verification_stats['targeted_correction_2']}")
print(f"  Claude 4.5 full custom generation: {verification_stats['full_custom_generation']}")
print(f"  Errors: {verification_stats['errors']}")

## Step 5: Create Output DataFrames

Convert verification results into structured DataFrames.

In [ ]:
# Create Output DataFrames
# Create DataFrame from verified results (full data)
if verified_results:
    verified_df = pd.DataFrame(verified_results)
    
    # Ensure LLM_used is last column
    cols = verified_df.columns.tolist()
    cols = [col for col in cols if col != 'LLM_used'] + ['LLM_used']
    verified_df = verified_df[cols]
    
    print(f"\n✓ Verified DataFrame (combined) shape: {verified_df.shape}")
    print(f"✓ Columns: {list(verified_df.columns)}")
    print(f"\nFirst verified row (LLM_used column):")
    print(verified_df[['game_id', 'round_id', 'LLM_used']].head())
else:
    print("No results to create verified output CSV")
    verified_df = None

# Create DataFrame from criteria results (detailed per-criterion tracking)
if criteria_results:
    criteria_df = pd.DataFrame(criteria_results)
    
    print(f"\n✓ Criteria DataFrame shape: {criteria_df.shape}")
    print(f"✓ Columns: {list(criteria_df.columns)}")
    print(f"\nFirst criteria records:")
    print(criteria_df.head())
    
    # Calculate aggregate statistics
    print(f"\n{'='*60}")
    print("AGGREGATE STATISTICS (for paper analysis):")
    print(f"{'='*60}")
    
    # Helper function to count passes
    def count_passes(df, prefix):
        roles = (df[f'{prefix}_Roles'] == 1).sum()
        history = (df[f'{prefix}_History'] == 1).sum()
        matrix = (df[f'{prefix}_Matrix'] == 1).sum()
        authenticity = (df[f'{prefix}_Authenticity'] == 1).sum()
        format_check = (df[f'{prefix}_Format'] == 1).sum()
        total = len(df)
        return {
            'Roles': f"{roles}/{total} ({roles/total*100:.1f}%)",
            'History': f"{history}/{total} ({history/total*100:.1f}%)",
            'Matrix': f"{matrix}/{total} ({matrix/total*100:.1f}%)",
            'Authenticity': f"{authenticity}/{total} ({authenticity/total*100:.1f}%)",
            'Format': f"{format_check}/{total} ({format_check/total*100:.1f}%)"
        }
    
    gemini31_stats = count_passes(criteria_df, 'Gemini31')
    gpt52_stats = count_passes(criteria_df, 'GPT52')
    
    print("\nGemini-3.1 Pass Rates:")
    for criterion, stat in gemini31_stats.items():
        print(f"  {criterion:15s}: {stat}")
    
    print("\nGPT-5.2 Pass Rates:")
    for criterion, stat in gpt52_stats.items():
        print(f"  {criterion:15s}: {stat}")
else:
    print("No results to create criteria output CSV")
    criteria_df = None

## Step 6: Output Files

Files saved by the verification process:

1. **verified_r1_seeds_combined.csv**: Full dialogue rows with columns:
   - All original columns from input CSVs
   - `LLM_used`: Source indicator (Gemini-3.1, GPT-5.2, Gemini-3.1-Claude-4.5, GPT-5.2-Claude-4.5, or Claude-4.5)
   - `Claude_Reasoning`: Claude's explanation for the selection decision

2. **verified_r1_criteria_scores.csv**: Detailed per-criterion validation tracking with columns:
   - `ID`: Game/Round identifier
   - `Gemini31_Roles`, `Gemini31_History`, `Gemini31_Matrix`, `Gemini31_Authenticity`, `Gemini31_Format`: Binary checks (1 = pass, 0 = fail)
   - `Gemini31_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `GPT52_Roles`, `GPT52_History`, `GPT52_Matrix`, `GPT52_Authenticity`, `GPT52_Format`: Binary checks (1 = pass, 0 = fail)
   - `GPT52_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `Selected_LLM`: Which model's output was used
   - `Correction_Mode`: None, Pairwise_Tiebreaker, Targeted, or Full_Custom
   - `Claude_Reasoning`: Claude's explanation for the decision
   - `Gemini31_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)
   - `GPT52_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)

**For Paper Analysis:** The criteria CSV enables reporting like: "Gemini-3.1 achieved 95% format compliance (214/225 games)" and identifying which criteria are most challenging for each model. Criteria explanations provide insights into why each criterion passed or failed.

In [ ]:
# Save Verified Results
if verified_df is not None and criteria_df is not None:
    # Save combined verified results
    verified_df.to_csv(OUTPUT_FILE_COMBINED, index=False)
    print(f"✓ Verified results saved to {OUTPUT_FILE_COMBINED}")
    print(f"  Total rows: {len(verified_df)}")
    
    # Save criteria tracking with detailed per-criterion scores
    criteria_df.to_csv(OUTPUT_FILE_CRITERIA, index=False)
    print(f"\n✓ Criteria tracking saved to {OUTPUT_FILE_CRITERIA}")
    print(f"  Total records: {len(criteria_df)}")
    
    # Display samples
    print(f"\nSample of verified results (LLM_used distribution):")
    print(verified_df['LLM_used'].value_counts())
    
    print(f"\nSample of criteria records (first 5):")
    print(criteria_df.head(5))
    
    # Display correction mode distribution
    print(f"\nCorrection mode distribution:")
    print(criteria_df['Correction_Mode'].value_counts())
else:
    print("No verified results to save")

## Next Steps: Full Verification

After testing this notebook with the first 2 games and reviewing the results:

1. **Review Sample Results**: Check that binary validation logic works correctly
2. **Check Criteria Distribution**: Examine which criteria pass/fail most frequently
3. **Validate Corrections**: 
   - Check "Targeted" corrections properly fix only the failing criterion
   - Verify "Full_Custom" generations when both models score ≤3/5
   - Confirm "Pairwise_Tiebreaker" selections when both score 5/5
4. **Verify Decision Logic**: Ensure the system correctly handles all scenarios:
   - Both 5/5 → Pairwise comparison
   - One 5/5, other <5 → Selects 5/5
   - At least one 4/5 → Targeted correction (defaults to GPT-5.2 if both 4/5)
   - Both ≤3/5 → Full custom generation
5. **Scale Up**: Change `TEST_LIMIT = 2` to `TEST_LIMIT = len(gemini3_df)` to process all 225 games
6. **Analyze Results**: Use criteria CSV to calculate per-criterion pass rates for your paper
7. **Final Merge**: Combine verified results with 25 manually generated seeds for complete 250-seed dataset

**Important Notes:**
- Current test uses TEST_LIMIT = 2 to validate the system
- Full verification of 225 games will take ~4-5 minutes (1 sec delay + longer max_tokens)
- Per-criterion tracking enables detailed analysis for paper (e.g., "Format: 98% pass rate")

## Comprehensive Analysis: Binary Multi-Criteria Validation Results

Detailed analysis of the verified r1 criteria scores including:
- Per-criterion passing rates for both LLMs
- Average performance metrics
- LLM selection statistics
- Correction mode distribution
- Most/least reliable criteria
- Additional insights

In [ ]:
# Load and Analyze Verified R1 Criteria Scores
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the criteria scores CSV
criteria_df = pd.read_csv('verified_r1_criteria_scores.csv')

print("="*80)
print("COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results")
print("="*80)
print(f"\nTotal Records Analyzed: {len(criteria_df)}")
print(f"Columns: {list(criteria_df.columns)}")

# ============================================================================
# 1. PER-CRITERION PASSING RATES FOR BOTH LLMs
# ============================================================================
print("\n" + "="*80)
print("1. PER-CRITERION PASSING RATES")
print("="*80)

criteria_names = ['Roles', 'History', 'Matrix', 'Authenticity', 'Format']

print("\nGemini-3.1 Passing Rates:")
gemini31_passing_rates = {}
for criterion in criteria_names:
    col_name = f'Gemini31_{criterion}'
    passing_rate = (criteria_df[col_name].sum() / len(criteria_df)) * 100
    gemini31_passing_rates[criterion] = passing_rate
    print(f"  {criterion:15s}: {passing_rate:6.2f}% ({criteria_df[col_name].sum()}/{len(criteria_df)})")

print("\nGPT-5.2 Passing Rates:")
gpt52_passing_rates = {}
for criterion in criteria_names:
    col_name = f'GPT52_{criterion}'
    passing_rate = (criteria_df[col_name].sum() / len(criteria_df)) * 100
    gpt52_passing_rates[criterion] = passing_rate
    print(f"  {criterion:15s}: {passing_rate:6.2f}% ({criteria_df[col_name].sum()}/{len(criteria_df)})")

# Comparison
print("\nComparative Analysis (GPT-5.2 vs Gemini-3.1):")
for criterion in criteria_names:
    diff = gpt52_passing_rates[criterion] - gemini31_passing_rates[criterion]
    winner = "GPT-5.2" if diff > 0 else ("Gemini-3.1" if diff < 0 else "TIE")
    print(f"  {criterion:15s}: {diff:+6.2f}% difference (Winner: {winner})")

# ============================================================================
# 2. AVERAGE PERFORMANCE OF BOTH LLMs
# ============================================================================
print("\n" + "="*80)
print("2. AVERAGE PERFORMANCE METRICS")
print("="*80)

# Calculate average scores (out of 5)
gemini31_scores = []
gpt52_scores = []

for idx, row in criteria_df.iterrows():
    gemini31_score = sum([row[f'Gemini31_{c}'] for c in criteria_names])
    gpt52_score = sum([row[f'GPT52_{c}'] for c in criteria_names])
    gemini31_scores.append(gemini31_score)
    gpt52_scores.append(gpt52_score)

gemini31_avg = np.mean(gemini31_scores)
gpt52_avg = np.mean(gpt52_scores)
gemini31_std = np.std(gemini31_scores)
gpt52_std = np.std(gpt52_scores)

print(f"\nGemini-3.1:")
print(f"  Average Score: {gemini31_avg:.3f}/5 (SD: {gemini31_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gemini31_scores.count(score)
    pct = (count / len(gemini31_scores)) * 100
    print(f"    {score}/5: {count:4d} games ({pct:5.2f}%)")

print(f"\nGPT-5.2:")
print(f"  Average Score: {gpt52_avg:.3f}/5 (SD: {gpt52_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gpt52_scores.count(score)
    pct = (count / len(gpt52_scores)) * 100
    print(f"    {score}/5: {count:4d} games ({pct:5.2f}%)")

print(f"\nOverall Comparison:")
if gpt52_avg > gemini31_avg:
    print(f"  GPT-5.2 outperforms Gemini-3.1 by {gpt52_avg - gemini31_avg:.3f} points ({((gpt52_avg - gemini31_avg) / gemini31_avg) * 100:.2f}%)")
elif gemini31_avg > gpt52_avg:
    print(f"  Gemini-3.1 outperforms GPT-5.2 by {gemini31_avg - gpt52_avg:.3f} points ({((gemini31_avg - gpt52_avg) / gpt52_avg) * 100:.2f}%)")
else:
    print(f"  Both models perform equally with average score of {gemini31_avg:.3f}/5")

# ============================================================================
# 3. LLM SELECTION STATISTICS
# ============================================================================
print("\n" + "="*80)
print("3. LLM SELECTION STATISTICS")
print("="*80)

# Parse Selected_LLM column to extract which model's output was selected
selection_counts = criteria_df['Selected_LLM'].value_counts()
print("\nSelection Breakdown:")
for selection, count in selection_counts.items():
    pct = (count / len(criteria_df)) * 100
    print(f"  {selection:30s}: {count:4d} ({pct:5.2f}%)")

# Extract which LLM was selected (Gemini-3.1, GPT-5.2, or Claude custom)
gemini31_selected = 0
gpt52_selected = 0
claude_custom = 0

for selection in criteria_df['Selected_LLM']:
    if 'Gemini-3.1' in selection and 'GPT-5.2' not in selection:
        gemini31_selected += 1
    elif 'GPT-5.2' in selection:
        gpt52_selected += 1
    elif 'Claude' in selection and 'GPT' not in selection and 'Gemini' not in selection:
        claude_custom += 1

print(f"\nBase Model Selection:")
print(f"  Gemini-3.1 selected:    {gemini31_selected:4d} ({(gemini31_selected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 selected:   {gpt52_selected:4d} ({(gpt52_selected/len(criteria_df))*100:5.2f}%)")
print(f"  Claude custom:      {claude_custom:4d} ({(claude_custom/len(criteria_df))*100:5.2f}%)")

# ============================================================================
# 4. CORRECTION MODE ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("4. CORRECTION MODE DISTRIBUTION")
print("="*80)

correction_counts = criteria_df['Correction_Mode'].value_counts()
print("\nCorrection Mode Breakdown:")
for mode, count in correction_counts.items():
    pct = (count / len(criteria_df)) * 100
    print(f"  {mode:20s}: {count:4d} ({pct:5.2f}%)")

# Analyze targeted corrections breakdown
targeted_selections = criteria_df[criteria_df['Correction_Mode'] == 'Targeted']
targeted_count = len(targeted_selections)

# Count BOTH score 4/5 (default to GPT-5.2)
both_4of5 = sum((g4 == 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

# Count only Gemini-3.1 scores 4/5
only_gemini31_4of5 = sum((g4 == 4 and g5 != 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

# Count only GPT-5.2 scores 4/5
only_gpt52_4of5 = sum((g4 != 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print(f"\nTargeted Correction Breakdown:")
print(f"  Total targeted corrections:          {targeted_count}")
print(f"  Both scored 4/5 (default GPT-5.2):   {both_4of5:4d} ({(both_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only Gemini-3.1 scored 4/5:              {only_gemini31_4of5:4d} ({(only_gemini31_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only GPT-5.2 scored 4/5:             {only_gpt52_4of5:4d} ({(only_gpt52_4of5/len(criteria_df))*100:5.2f}%)")

# Count Claude's own corrections vs using LLM outputs
claude_corrections = criteria_df[criteria_df['Correction_Mode'].str.contains('Claude', na=False)]
print(f"\nClaude's Own Corrections:")
print(f"  Full custom generations: {len(criteria_df[criteria_df['Correction_Mode'] == 'Full_Custom']):4d}")
print(f"  Targeted corrections:    {len(criteria_df[criteria_df['Correction_Mode'] == 'Targeted']):4d}")

# ============================================================================
# 5. MOST FAILING AND PASSING CRITERIA
# ============================================================================
print("\n" + "="*80)
print("5. MOST FAILING AND PASSING CRITERIA")
print("="*80)

# Sort criteria by passing rate
gemini31_sorted = sorted(gemini31_passing_rates.items(), key=lambda x: x[1])
gpt52_sorted = sorted(gpt52_passing_rates.items(), key=lambda x: x[1])

print("\nGemini-3.1:")
print(f"  Most Challenging: {gemini31_sorted[0][0]} ({gemini31_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gemini31_sorted[-1][0]} ({gemini31_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst to best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gemini31_sorted])}")

print("\nGPT-5.2:")
print(f"  Most Challenging: {gpt52_sorted[0][0]} ({gpt52_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gpt52_sorted[-1][0]} ({gpt52_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst to best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gpt52_sorted])}")

# ============================================================================
# 5a. DETAILED SELECTION BREAKDOWN
# ============================================================================
print("\n" + "="*80)
print("5a. DETAILED SELECTION BREAKDOWN")
print("="*80)

# Count based on actual Selected_LLM values (not scores)
gemini31_direct = selection_counts.get('Gemini-3.1', 0)
gemini31_corrected = selection_counts.get('Gemini-3.1-Claude-4.5', 0)
gpt52_direct = selection_counts.get('GPT-5.2', 0)
gpt52_corrected = selection_counts.get('GPT-5.2-Claude-4.5', 0)
claude_full_custom = selection_counts.get('Claude-4.5', 0)

print("\nHow Often Each Model's Output Was Used:")
print(f"  Gemini-3.1 selected directly:            {gemini31_direct:4d} ({(gemini31_direct/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 corrected by Claude:          {gemini31_corrected:4d} ({(gemini31_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 total usage:                  {gemini31_direct + gemini31_corrected:4d} ({((gemini31_direct + gemini31_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  GPT-5.2 selected directly:           {gpt52_direct:4d} ({(gpt52_direct/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 corrected by Claude:         {gpt52_corrected:4d} ({(gpt52_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 total usage:                 {gpt52_direct + gpt52_corrected:4d} ({((gpt52_direct + gpt52_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  Claude full custom (both ≤3/5):      {claude_full_custom:4d} ({(claude_full_custom/len(criteria_df))*100:5.2f}%)")
print(f"  Pairwise tiebreaker (both 5/5):      {both_perfect:4d} ({(both_perfect/len(criteria_df))*100:5.2f}%)")

# ============================================================================
# 6. ADDITIONAL INSIGHTS
# ============================================================================
print("\n" + "="*80)
print("6. ADDITIONAL INSIGHTS")
print("="*80)

# Both models perfect (5/5)
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
print(f"\nBoth models scored 5/5: {both_perfect:4d} games ({(both_perfect/len(criteria_df))*100:5.2f}%)")

# Both models failed same criterion
print("\nCriteria where both models commonly fail together:")
for criterion in criteria_names:
    both_fail = sum((criteria_df[f'Gemini31_{criterion}'] == 0) & (criteria_df[f'GPT52_{criterion}'] == 0))
    pct = (both_fail / len(criteria_df)) * 100
    print(f"  {criterion:15s}: {both_fail:4d} games ({pct:5.2f}%)")

# One perfect, one not
one_perfect = sum((g4 == 5) != (g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
print(f"\nExactly one model scored 5/5: {one_perfect:4d} games ({(one_perfect/len(criteria_df))*100:5.2f}%)")

# Both models scored ≤3/5 (requiring full custom generation)
both_low = sum((g4 <= 3 and g5 <= 3) for g4, g5 in zip(gemini31_scores, gpt52_scores))
print(f"\nBoth models scored ≤3/5: {both_low:4d} games ({(both_low/len(criteria_df))*100:5.2f}%)")

# Agreement rate (both score same)
agreement = sum(g4 == g5 for g4, g5 in zip(gemini31_scores, gpt52_scores))
print(f"\nScore Agreement: {agreement:4d} games ({(agreement/len(criteria_df))*100:5.2f}%) where both models got same total score")

# Per-criterion agreement
print("\nPer-Criterion Agreement (both pass or both fail):")
for criterion in criteria_names:
    agree_count = sum(criteria_df[f'Gemini31_{criterion}'] == criteria_df[f'GPT52_{criterion}'])
    pct = (agree_count / len(criteria_df)) * 100
    print(f"  {criterion:15s}: {agree_count:4d} agreements ({pct:5.2f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)